# RoadQure: which roads should a city repair first?

*Qiskit FallFest at Florida Gulf Coast University. Quantum optimization (QAOA) with Qiskit, checked against a classical solver.*

> **In one paragraph.** In South Florida, king tides and a rising water table soak the ground under roads, so they fail **from below** before a pothole ever shows.
> Cities repair what residents complain about, so these hidden failures get missed, often on **evacuation and hospital routes**.
> RoadQure scores every road for hidden damage, then asks: *with a fixed budget, which combination of repairs protects the most?*
> That is a combinatorial optimization problem. We write it as a QUBO, solve it with the **Quantum Approximate Optimization Algorithm (QAOA)** in Qiskit,
> and compare against an exact classical solver, a random baseline and "fix the most-complained-about roads."

### What this notebook does and does not claim
| We show | We do **not** claim |
|---|---|
| A real problem turned into a QUBO and solved with QAOA in Qiskit (`QAOAAnsatz`, `EstimatorV2`, `SamplerV2`, transpilation) | A quantum speedup. At 8 qubits a classical computer is instant |
| QAOA finds good plans far more often than random guessing, and its best plan is compared to the exact optimum | That these results describe the real Miami Beach: the data is **illustrative** (see section 2) |
| Simulator results, including a noise model taken from a real IBM device and error suppression | That the noisy simulator *is* hardware. A real-hardware run is a separate section (9) |

### How this maps to the judging criteria
| Question a judge asks | Where to look |
|---|---|
| Is the problem clear and does it matter? | Section 1 |
| Why might quantum help, and what are today's limits? | Sections 3 and 11 |
| Does the code run, and is Qiskit used appropriately? | Sections 4 to 8 |
| Are results shown against a classical baseline, and simulator vs. noise vs. hardware? | Sections 5, 8, 9 and 10 |
| What did we learn and what is next? | Section 11 |

**To run:** `pip install -r requirements.txt`, then run all cells top to bottom (about 3 minutes). Tables and charts are saved to `results/`.

## 1. The problem

**Who it affects.** Residents who depend on evacuation routes and hospital access, emergency managers, and city public-works teams in low-lying coastal cities. Miami Beach is our demo area.

**Why it matters.** Sea-level rise lifts the water table, and king tides push salt water into the ground under roads. Saturated road bases lose strength and fail from below, often repeatedly, even after the surface is patched.
A city with a limited repair budget has to choose which roads to fix. Choosing by complaint count favors the roads people can *see*, not the ones quietly failing.

**The decision.** Given a budget, choose the set of road segments to repair so that the most important routes stay open after the next king tide.

## 2. The data (illustrative, and labeled that way)

Each row is a road segment. This data was prepared by a teammate for the demo (`data/segments.csv`, notes in `data/DATA_NOTES.md`).

| Real | Estimated or simulated |
|---|---|
| Street names, the official Collins Ave evacuation route, the location of Mount Sinai Medical Center | Coordinates (approximate), elevation, tide exposure, groundwater depth, 311 complaint counts, repair costs, which roads lead to the hospital |

So the results below show that **the method works**, not what Miami Beach should actually repair. Swapping in measured data needs no code changes, only a new `data/segments.csv`.

**How a road gets its numbers.** `risk` (0 to 1) combines tide exposure (weight 0.45), shallow groundwater (0.35) and low elevation (0.20), then squashes the result so it behaves like a failure probability.
`criticality` is 1, plus 1.5 on an evacuation route, plus 1.0 on the hospital route. `value = risk x criticality` is the expected harm avoided by repairing the road.
**Size knobs:** `K` roads go to the quantum optimizer (a classical step shortlists the `K` most valuable), and the lean formulation uses one qubit per road, so qubits = `K`.

In [1]:
import sys, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "results"; RESULTS.mkdir(exist_ok=True)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler import generate_preset_pass_manager, PassManager
from qiskit.transpiler.passes import ALAPScheduleAnalysis, PadDynamicalDecoupling
from qiskit.circuit.library import XGate
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2, SamplerV2
from qiskit_optimization.translators import to_ising
from qiskit_ibm_runtime.fake_provider import FakeTorino
from qiskit_optimization.converters import QuadraticProgramToQubo

from undertow.data_synthetic import make_grid_network, adjacency_pairs
from undertow.risk import add_scores
from undertow.qubo import build_program, build_lean_program, tune_price, plan_value
from undertow.solve import solve_exact, solve_current_policy
from undertow.metrics import score_plan
from undertow.charts import evac_route_chart, good_plans_chart, training_chart, network_figure
from IPython.display import Markdown, display

# Chart style: thin marks, recessive grid, text in neutral ink
BLUE, ORANGE, AQUA, GRAY, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#9a9893", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#d6d5d0",
    "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "axes.grid": True, "grid.color": "#ebeae6", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11, "axes.titlesize": 13,
    "axes.titleweight": "bold", "axes.titlelocation": "left",
})
SEED = 7
from solver.scenario import HOSPITAL   # Mount Sinai Medical Center (approximate location)

In [2]:
K, BUDGET, REPS = 8, 14, 2   # same settings as the web app: 8 roads -> 8 qubits, 14 crew-days (Medium), 2 QAOA layers

csv = ROOT / "data" / "segments.csv"
df = pd.read_csv(csv) if csv.exists() else make_grid_network()[1]
print("data:", csv.name if csv.exists() else "synthetic network")
df = add_scores(df)
pairs_all = adjacency_pairs(df)

# Hybrid step: the classical risk model shortlists the K most valuable candidates.
cand = df.sort_values("value", ascending=False).head(K)
ids = set(cand.segment_id)
pairs = [(a, b) for a, b in pairs_all if a in ids and b in ids]
print(f"{len(df)} segments in the area, {K} candidates sent to the optimizer, budget = {BUDGET} crew-days")

geometry = json.loads((ROOT / "data" / "segments_geometry.json").read_text()) if (ROOT / "data" / "segments_geometry.json").exists() else {}
context = json.loads((ROOT / "data" / "street_context.json").read_text())["ways"] if (ROOT / "data" / "street_context.json").exists() else []
cand[["segment_id", "street_name", "risk", "criticality", "value", "repair_cost", "reports_311", "evac_route", "hospital_access"]]

data: segments.csv
20 segments in the area, 8 candidates sent to the optimizer, budget = 14 crew-days


,segment_id,street_name,risk,criticality,value,repair_cost,reports_311,evac_route,hospital_access
12,S13,Alton Rd,0.909,2.0,1.818,3,3,False,True
4,S05,41st St,0.873,2.0,1.746,3,5,False,True
13,S14,Alton Rd,0.804,2.0,1.608,4,2,False,True
5,S06,41st St,0.599,2.0,1.198,4,3,False,True
6,S07,41st St,0.599,2.0,1.198,5,1,False,True
0,S01,W 40th St,0.936,1.0,0.936,3,3,False,False
1,S02,W 40th St,0.755,1.0,0.755,4,1,False,False
2,S03,W 40th St,0.755,1.0,0.755,5,5,False,False


In [3]:
fig = network_figure(df, geometry, hospital=HOSPITAL, context=context, title="Hidden damage risk: brighter orange = more likely failing underneath", label_chosen=False)
fig.savefig(RESULTS / "network_risk.png", dpi=150, facecolor=fig.get_facecolor())
fig

<Figure size 1100x520 with 1 Axes>

## 3. From a repair plan to qubits

Let `x_i = 1` mean "repair road `i`". We want the plan that avoids the most harm within the budget `B`:

$$\max \sum_i v_i x_i + \sum_{(i,j)\ \text{connected}} s_{ij}\, x_i x_j \quad \text{subject to} \quad \sum_i c_i x_i \le B$$

- `v_i` is the harm avoided by repairing road `i`, and `c_i` is its cost in crew-days.
- `s_ij` is a bonus for repairing two **connected** roads together, larger when both are on the same evacuation or hospital route. Fixing one block of a route is worth little if the next block still floods.

**Why this suits quantum optimization.** A function of binary variables with at most pairwise terms is a **QUBO**. Substituting `x_i = (1 - Z_i) / 2` turns it into an Ising Hamiltonian `H_C` made of `Z` and `ZZ` terms
(Qiskit does this with `to_ising`), whose lowest-energy state is the best plan. The pairwise term is what makes the problem hard and what QAOA is designed for.

**The search space.** With `n` roads there are `2^n` possible plans:

In [4]:
rows = [(n_, f"{2 ** n_:,}") for n_ in (8, 20, 40, 60)]
pd.DataFrame(rows, columns=["roads considered", "possible repair plans (2^n)"]).set_index("roads considered")

,possible repair plans (2^n)
roads considered,
8,256
20,"1,048,576"
40,"1,099,511,627,776"
60,"1,152,921,504,606,846,976"


**Encoding the budget.** We tried two ways:
- **Slack (first attempt):** Qiskit's `QuadraticProgramToQubo` turns the budget inequality into a penalty using extra *slack qubits*. It is exact, but costs more qubits and gives the shallow circuit a harder landscape.
- **Lean (what we use):** one qubit per road. The budget becomes a **price per crew-day** `λ`, found by bisection (the lowest price whose best plan fits the budget). Plans that still exceed the budget are discarded after sampling (**post-selection**).

In [5]:
qp = build_program(cand, pairs, BUDGET)             # the true constrained problem (for the exact baseline)
price = tune_price(cand, pairs, BUDGET)
lean = build_lean_program(cand, pairs, price)
cost_op, offset = to_ising(lean)
n = cost_op.num_qubits
slack_conv = QuadraticProgramToQubo()
slack_op, slack_offset = slack_conv.convert(qp).to_ising()
print(qp.export_as_lp_string())
print(f"price per cost unit λ = {price:.3f}")
print(f"lean: {n} qubits, {len(cost_op)} Pauli terms | slack: {slack_op.num_qubits} qubits, {len(slack_op)} Pauli terms")

\ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: undertow_repair_plan

Maximize
 obj: 1.818000000000 S13 + 1.746000000000 S05 + 1.608000000000 S14
      + 1.198000000000 S06 + 1.198000000000 S07 + 0.936000000000 S01
      + 0.755000000000 S02 + 0.755000000000 S03 + [ 1.746000000000 S13*S05
      + 1.608000000000 S13*S14 + 0.454000000000 S13*S01 + 1.608000000000 S05*S14
      + 1.198000000000 S05*S06 + 1.198000000000 S06*S07 + 0.378000000000 S01*S02
      + 0.378000000000 S02*S03 ]/2
Subject To
 budget: 3 S13 + 3 S05 + 4 S14 + 4 S06 + 5 S07 + 3 S01 + 4 S02 + 5 S03 <= 14

Bounds
 0 <= S13 <= 1
 0 <= S05 <= 1
 0 <= S14 <= 1
 0 <= S06 <= 1
 0 <= S07 <= 1
 0 <= S01 <= 1
 0 <= S02 <= 1
 0 <= S03 <= 1

Binaries
 S13 S05 S14 S06 S07 S01 S02 S03
End

price per cost unit λ = 0.388
lean: 8 qubits, 16 Pauli terms | slack: 12 qubits, 78 Pauli terms


## 4. Baselines

Every result is compared to three things:
- **Exact classical optimum:** a brute-force eigensolver over all `2^n` plans. At this size it is instant and always right.
- **Today's approach:** repair the roads with the most 311 complaints until the budget runs out.
- **Random guessing:** the chance a uniformly random bitstring is a good plan. QAOA has to beat this to be doing anything useful.

In [6]:
exact = solve_exact(qp)
policy = solve_current_policy(df, BUDGET)
best_value = plan_value(cand, pairs, exact.chosen)

costs = cand.set_index("segment_id")["repair_cost"].to_dict()
names = [v.name for v in lean.variables]

def decode(bitstring, var_names=names):
    # Qiskit bitstrings are little-endian: the rightmost bit is qubit 0
    chosen = {nm for nm, b in zip(var_names, bitstring[::-1]) if b == "1" and nm in costs}  # drop slack qubits
    return chosen, sum(costs[s] for s in chosen) <= BUDGET

def evaluate(counts, var_names=names):
    total = sum(counts.values())
    feasible = optimal = good = 0
    best = 0.0
    for bs, c in counts.items():
        chosen, ok = decode(bs, var_names)
        if ok:
            v = plan_value(cand, pairs, chosen)
            feasible += c
            best = max(best, v)
            optimal += c * (chosen == exact.chosen)
            good += c * (v >= 0.9 * best_value)
    return {"P(feasible plan)": feasible / total, "P(optimal plan)": optimal / total,
            "P(good plan, >=90% of optimum)": good / total,
            "best plan found": round(best, 3), "best / optimum": round(best / best_value, 3)}

# Random guessing: every bitstring equally likely
uniform = {format(i, f"0{n}b"): 1 for i in range(2 ** n)}   # every possible plan equally likely
random_row = evaluate(uniform)
print("Optimal plan:", sorted(exact.chosen), "value", best_value)
print("Current policy picks:", sorted(policy.chosen))
print(f"A random plan is optimal {random_row['P(optimal plan)']:.2%} of the time and good {random_row['P(good plan, >=90% of optimum)']:.2%} of the time")

Optimal plan: ['S05', 'S06', 'S13', 'S14'] value 9.45
Current policy picks: ['S03', 'S05', 'S08', 'S10']
A random plan is optimal 0.39% of the time and good 0.78% of the time


## 5. The QAOA circuit

QAOA starts every qubit in an equal superposition, then alternates two layers, `REPS` times:
- a **cost layer** `exp(-i γ H_C)`: phase rotations from our Hamiltonian, which give good plans a different phase, and
- a **mixer layer** `exp(-i β Σ X)`: rotations that let probability flow between plans.

A classical optimizer tunes the angles `γ` and `β` so that measuring the circuit returns low-energy (good) plans more often. This is a **hybrid** quantum-classical loop.
We build the circuit explicitly with `QAOAAnsatz`. The full circuit is too wide to read, so the diagram uses a 3-road toy version of the same problem.

In [7]:
ansatz = QAOAAnsatz(cost_op, reps=REPS)
print(f"QAOA ansatz: {ansatz.num_qubits} qubits, {ansatz.num_parameters} parameters (γ, β per layer)")

# Readable toy circuit for the slides
toy = cand.head(3)
toy_ids = set(toy.segment_id)
toy_op, _ = to_ising(build_lean_program(toy, [(a, b) for a, b in pairs if a in toy_ids and b in toy_ids], price))
toy_circ = QAOAAnsatz(toy_op, reps=1).decompose(reps=2)
fig = toy_circ.draw("mpl", fold=-1, style="iqp")
fig.savefig(RESULTS / "qaoa_circuit_toy.png", dpi=160, bbox_inches="tight")
fig

QAOA ansatz: 8 qubits, 4 parameters (γ, β per layer)


<Figure size 1374.44x284.278 with 1 Axes>

## 6. Transpilation for a real IBM device

Real hardware has a limited set of native gates (for IBM Heron: `cz`, `rz`, `sx`, `x`) and limited qubit connectivity.
The transpiler rewrites our circuit to fit, which adds depth and two-qubit gates. More of them means more noise.
We compare optimization levels 0 to 3 for **FakeTorino**, an offline snapshot of a real 133-qubit IBM Heron device (its coupling map, gate set and calibration data).

In [8]:
backend = FakeTorino()
measured = ansatz.measure_all(inplace=False)

rows = []
isa = {}
for level in range(4):
    pm = generate_preset_pass_manager(optimization_level=level, backend=backend, seed_transpiler=SEED)
    t = time.perf_counter()
    circ = pm.run(measured)
    ops = circ.count_ops()
    isa[level] = circ
    rows.append({"optimization level": level, "depth": circ.depth(),
                 "two-qubit gates": ops.get("cz", 0) + ops.get("ecr", 0) + ops.get("cx", 0),
                 "transpile seconds": round(time.perf_counter() - t, 2)})
transpile_table = pd.DataFrame(rows).set_index("optimization level")
transpile_table.to_csv(RESULTS / "transpilation.csv")
print(f"Before transpiling: depth {measured.decompose(reps=3).depth()} in abstract gates")
transpile_table

Before transpiling: depth 33 in abstract gates


,depth,two-qubit gates,transpile seconds
optimization level,,,
0,199,65,0.01
1,128,41,0.01
2,94,37,0.01
3,93,37,0.01


**What this means:** a circuit that looks small on paper grows when it is fitted to real hardware. Fewer two-qubit gates means less noise and a more reliable answer, so compilation is itself a form of error suppression.

## 7. Training the angles with `EstimatorV2`

The classical optimizer (COBYLA) proposes angles, and the **Estimator primitive** returns the expected energy `⟨ψ(γ,β)| H_C |ψ(γ,β)⟩`. Lower energy means better plans.
We train on the ideal simulator (`qiskit_aer.primitives.EstimatorV2`) with 3 random starts per formulation, then sample with `SamplerV2`. There is no CVaR or other special objective, just the plain expectation value.
On real hardware this loop would run inside a Qiskit Runtime session. This is also where we compare the slack and lean encodings.

In [9]:
sim = AerSimulator(seed_simulator=SEED)
sim_pm = generate_preset_pass_manager(optimization_level=1, backend=sim)
estimator = EstimatorV2(options={"run_options": {"seed": SEED}})
ideal_sampler = SamplerV2(seed=SEED)

def train(op, reps, starts=3, maxiter=200):
    circ = sim_pm.run(QAOAAnsatz(op, reps=reps))
    histories, best = [], None
    for start in range(starts):
        hist = []
        def energy(theta):
            e = float(estimator.run([(circ, op, theta)]).result()[0].data.evs)
            hist.append(e)
            return e
        x0 = np.random.default_rng(start).uniform(0, np.pi, circ.num_parameters)
        res = minimize(energy, x0, method="COBYLA", options={"maxiter": maxiter})
        histories.append(hist)
        if best is None or res.fun < best.fun:
            best = res
    return best.x, histories

t = time.perf_counter()
theta_opt, histories = train(cost_op, REPS)
theta_slack, _ = train(slack_op, REPS)
print(f"trained both formulations in {time.perf_counter() - t:.1f}s")

# Slack vs lean on the ideal simulator
slack_names = [v.name for v in slack_conv.convert(qp).variables]
def sample_ideal(op, theta, reps):
    c = sim_pm.run(QAOAAnsatz(op, reps=reps).measure_all(inplace=False)).assign_parameters(theta)
    return ideal_sampler.run([c], shots=4000).result()[0].data.meas.get_counts()
cmp_rows = []
for label, op, th, vn in [("Slack (first attempt)", slack_op, theta_slack, slack_names), ("Lean (one qubit per segment)", cost_op, theta_opt, names)]:
    l3 = generate_preset_pass_manager(optimization_level=3, backend=FakeTorino(), seed_transpiler=SEED).run(
        QAOAAnsatz(op, reps=REPS).measure_all(inplace=False))
    ev = evaluate(sample_ideal(op, th, REPS), vn)
    cmp_rows.append({"formulation": label, "qubits": op.num_qubits,
                     "two-qubit gates on device": l3.count_ops().get("cz", 0),
                     "P(good plan), ideal": ev["P(good plan, >=90% of optimum)"],
                     "x better than random": round(ev["P(good plan, >=90% of optimum)"] / random_row["P(good plan, >=90% of optimum)"], 1)})
formulations = pd.DataFrame(cmp_rows).set_index("formulation")
formulations.to_csv(RESULTS / "formulation_comparison.csv")

fig = training_chart(histories)
fig.savefig(RESULTS / "qaoa_training.png", dpi=160, bbox_inches="tight")
formulations.style.format({"P(good plan), ideal": "{:.1%}"})

trained both formulations in 6.6s


,qubits,two-qubit gates on device,"P(good plan), ideal",x better than random
formulation,,,,
Slack (first attempt),12,468,2.2%,2.800000
Lean (one qubit per segment),8,37,24.6%,31.500000


**What this means:** the table compares the two ways of writing the budget. The lean encoding needs fewer qubits and far fewer two-qubit gates, and it works much better. The chart shows the optimizer's energy improving over its steps.

## 8. Sampling with `SamplerV2`: ideal vs. noisy vs. error suppression

The same trained circuit, transpiled for FakeTorino, is run four ways with 4,000 shots each. The "noisy" runs use Qiskit Aer with a **noise model built from the FakeTorino snapshot**
(gate errors, readout errors, T1/T2 relaxation). That is a simulation of a real device's noise, **not** a run on hardware.

| Run | What it models |
|---|---|
| Ideal | A perfect quantum computer (noiseless simulation of the device-ready circuit) |
| Noisy, level 1 | The device's noise model with light compilation |
| Noisy, level 3 | The same noise model with the heaviest compilation: fewer gates, less noise (**error suppression by compilation**) |
| Noisy, level 3 + DD | Adds **dynamical decoupling**: pairs of X pulses on idle qubits to cancel slow dephasing |

Every run is **post-selected**: bitstrings that break the budget are thrown away, and "best plan found" is the best of the rest.

In [10]:
noisy_sampler = SamplerV2.from_backend(backend, seed=SEED)
ideal_sampler = SamplerV2(seed=SEED)

dd_pm = generate_preset_pass_manager(optimization_level=3, backend=backend, seed_transpiler=SEED)
dd_pm.scheduling = PassManager([
    ALAPScheduleAnalysis(target=backend.target),
    PadDynamicalDecoupling(target=backend.target, dd_sequence=[XGate(), XGate()]),
])

runs = {
    "Ideal simulator": (ideal_sampler, isa[1]),
    "Noisy, level 1": (noisy_sampler, isa[1]),
    "Noisy, level 3": (noisy_sampler, isa[3]),
    "Noisy, level 3 + DD": (noisy_sampler, dd_pm.run(measured)),
}
rows, counts_by_run = [], {}
for name, (sampler, circ) in runs.items():
    t = time.perf_counter()
    counts = sampler.run([circ.assign_parameters(theta_opt)], shots=4000).result()[0].data.meas.get_counts()
    counts_by_run[name] = counts
    rows.append({"run": name, **evaluate(counts), "seconds": round(time.perf_counter() - t, 1)})
rows.append({"run": "Random guessing", **random_row, "seconds": 0})
results = pd.DataFrame(rows).set_index("run")
results["x better than random"] = (results["P(good plan, >=90% of optimum)"] / random_row["P(good plan, >=90% of optimum)"]).round(1)
results.to_csv(RESULTS / "sampling_results.csv")
results.style.format({"P(feasible plan)": "{:.1%}", "P(optimal plan)": "{:.2%}", "P(good plan, >=90% of optimum)": "{:.1%}"})

,P(feasible plan),P(optimal plan),"P(good plan, >=90% of optimum)",best plan found,best / optimum,seconds,x better than random
run,,,,,,,
Ideal simulator,45.0%,15.20%,24.5%,9.450000,1.000000,0.100000,31.400000
"Noisy, level 1",48.3%,11.65%,18.8%,9.450000,1.000000,1.400000,24.000000
"Noisy, level 3",46.8%,12.53%,20.4%,9.450000,1.000000,1.400000,26.100000
"Noisy, level 3 + DD",47.5%,12.20%,20.0%,9.450000,1.000000,3.600000,25.600000
Random guessing,41.8%,0.39%,0.8%,9.450000,1.000000,0.000000,1.000000


**What this means:** each row is a different way of running the same optimizer. The higher the "x better than random" number, the more often it lands on a good repair plan.

In [11]:
p_good = (results["P(good plan, >=90% of optimum)"] * 100).to_dict()
fig = good_plans_chart(p_good)
fig.savefig(RESULTS / "p_good_plan.png", dpi=160, bbox_inches="tight")
good_plans_chart(p_good, slide=True).savefig(RESULTS / "slide_good_plans_vs_random.png", dpi=120)  # 1920x1080 for slides
fig

<Figure size 950x440 with 1 Axes>

**What this means:** a random guess almost never lands on a good plan. The quantum search does so many times more often, and keeps most of that advantage under the noise of a real chip.

## 9. Real quantum hardware

The runs above used simulators. `scripts/run_hardware.py` takes the **same circuit and trained angles the web app uses**, transpiles it for a real IBM device, runs it once, and also runs the identical circuit on an ideal and a noisy simulator,
so the three can be compared side by side. If `results/hardware_run.json` exists, the cell below shows the comparison.

In [12]:
hw_path = RESULTS / "hardware_run.json"
if not hw_path.exists():
    display(Markdown("> **No real-hardware run is included in this notebook yet.** Everything above ran on simulators. "
                     "To add a real IBM Quantum run: `python scripts/run_hardware.py`, then re-run this cell."))
else:
    hw = json.loads(hw_path.read_text())
    assert hw["budget_units"] == BUDGET, "hardware run used a different budget than this notebook"
    rows = []
    for label, key in [(f"Real hardware: {hw['backend']}", "hardware"), ("Noisy simulator (FakeTorino snapshot)", "noisy_simulator"), ("Ideal simulator", "ideal_simulator")]:
        ev = evaluate(hw["counts"][key], hw["variable_names"])
        rows.append({"run": label, "P(feasible plan)": ev["P(feasible plan)"], "P(optimal plan)": ev["P(optimal plan)"],
                     "P(good plan, >=90% of optimum)": ev["P(good plan, >=90% of optimum)"], "best / optimum": ev["best / optimum"],
                     "x better than random": round(ev["P(good plan, >=90% of optimum)"] / random_row["P(good plan, >=90% of optimum)"], 1)})
    hw_table = pd.DataFrame(rows).set_index("run")
    display(Markdown(f"**Job `{hw['job_id']}` on `{hw['backend']}`**, {hw['shots']} shots, {hw['num_qubits']} qubits, "
                     f"{hw['transpiled']['two_qubit_gates']} two-qubit gates after transpiling (depth {hw['transpiled']['depth']}), run {hw['timestamp_utc']}."))
    display(hw_table.style.format({"P(feasible plan)": "{:.1%}", "P(optimal plan)": "{:.2%}", "P(good plan, >=90% of optimum)": "{:.1%}"}))

> **No real-hardware run is included in this notebook yet.** Everything above ran on simulators. To add a real IBM Quantum run: `python scripts/run_hardware.py`, then re-run this cell.

## 10. The answer that matters: the repair plan

Every plan is scored the same way on the **whole** area, not only the shortlist. A route counts as "open" only if every block of it survives the next king tide (failures treated as independent, and a repaired road keeps a 5% residual failure chance).
On this data the evacuation route sits on a low-risk road, so no plan changes it. The route to the hospital is where the plans differ, so that is what the chart shows.

In [13]:
def best_sampled_plan(counts):
    best, plan = -1, set()
    for bs in counts:
        chosen, ok = decode(bs)
        if ok and (v := plan_value(cand, pairs, chosen)) > best:
            best, plan = v, chosen
    return plan

plans = {
    "Current policy (most 311 reports)": policy.chosen,
    "Classical exact": exact.chosen,
    "QAOA, ideal simulator": best_sampled_plan(counts_by_run["Ideal simulator"]),
    "QAOA, noisy (level 3)": best_sampled_plan(counts_by_run["Noisy, level 3"]),
}
plan_table = pd.DataFrame({k: score_plan(df, pairs_all, v) for k, v in plans.items()}).T
plan_table["repairs"] = [" ".join(sorted(v)) for v in plans.values()]
plan_table.to_csv(RESULTS / "plan_comparison.csv")

# The route measure the optimized plan improves most (on this data that is the hospital route; the evacuation route is on a low-risk road)
METRICS = {"evacuation route": "evac_route_open_%", "route to the hospital": "hospital_access_open_%"}
lead_name, lead_col = max(METRICS.items(), key=lambda kv: plan_table.loc["QAOA, ideal simulator", kv[1]] - plan_table.loc["Current policy (most 311 reports)", kv[1]])
lead = plan_table[lead_col].astype(float).to_dict()
fig = evac_route_chart(lead, route=lead_name)
fig.savefig(RESULTS / "route_open.png", dpi=160, bbox_inches="tight")
evac_route_chart(lead, slide=True, route=lead_name).savefig(RESULTS / "slide_route_open.png", dpi=120)  # 1920x1080 for slides
plan_table[["segments_repaired", "cost_used", "evac_route_open_%", "hospital_access_open_%", "hidden_risk_removed", "repairs"]]

,segments_repaired,cost_used,evac_route_open_%,hospital_access_open_%,hidden_risk_removed,repairs
Current policy (most 311 reports),4.0,14.0,61.5,0.3,2.20,S03 S05 S08 S10
Classical exact,4.0,14.0,61.5,32.7,3.19,S05 S06 S13 S14
"QAOA, ideal simulator",4.0,14.0,61.5,32.7,3.19,S05 S06 S13 S14
"QAOA, noisy (level 3)",4.0,14.0,61.5,32.7,3.19,S05 S06 S13 S14


In [14]:
fig = network_figure(df, geometry, chosen=plans["QAOA, ideal simulator"], today=plans["Current policy (most 311 reports)"] - plans["QAOA, ideal simulator"],
                     hospital=HOSPITAL, context=context,
                     title="White: repairs chosen by the quantum search.  Violet: what today's approach fixes instead.")
fig.savefig(RESULTS / "network_plan.png", dpi=150, facecolor=fig.get_facecolor())
fig

<Figure size 1100x520 with 1 Axes>

**What this means:** with the same budget, the plan chosen by the quantum search keeps far more of the route to the hospital open than fixing whatever gets the most complaints. The exact classical solver is the reference for the best possible plan.

## 11. Results at a glance, honest limits and what is next

In [15]:
r = results
ideal_x, noisy1_x, noisy3_x, dd_x = (r.loc[k, "x better than random"] for k in ["Ideal simulator", "Noisy, level 1", "Noisy, level 3", "Noisy, level 3 + DD"])
pol, qa, ex = (plan_table.loc[k] for k in ["Current policy (most 311 reports)", "QAOA, ideal simulator", "Classical exact"])
same_as_exact = plans["QAOA, ideal simulator"] == plans["Classical exact"]
lines = [
    f"- **Quantum search vs. random guessing.** On an ideal simulator, {r.loc['Ideal simulator', 'P(good plan, >=90% of optimum)']:.1%} of QAOA samples are good repair plans "
    f"({ideal_x}x random guessing). With the noise model of a real IBM chip it falls to {noisy1_x}x (light compilation) or {noisy3_x}x (heavier compilation, "
    f"{transpile_table.loc[3, 'two-qubit gates']} vs {transpile_table.loc[1, 'two-qubit gates']} two-qubit gates); adding dynamical decoupling gives {dd_x}x.",
    f"- **Best plan found.** With post-selection, the best plan sampled reaches {r.loc['Ideal simulator', 'best / optimum']:.0%} of the optimum on the ideal simulator "
    f"and {r.loc['Noisy, level 3', 'best / optimum']:.0%} under noise.",
    f"- **Encoding the budget.** The lean encoding uses {formulations.loc['Lean (one qubit per segment)', 'qubits']} qubits and {formulations.loc['Lean (one qubit per segment)', 'two-qubit gates on device']} two-qubit gates "
    f"({formulations.loc['Lean (one qubit per segment)', 'x better than random']}x random guessing) versus {formulations.loc['Slack (first attempt)', 'qubits']} qubits and "
    f"{formulations.loc['Slack (first attempt)', 'two-qubit gates on device']} gates ({formulations.loc['Slack (first attempt)', 'x better than random']}x) for the slack encoding.",
    f"- **Against the classical baseline.** The quantum search's best plan {'is the same plan as' if same_as_exact else 'differs from'} the exact classical optimum "
    f"(objective {qa['objective']:.2f} vs {ex['objective']:.2f}).",
    f"- **Against today's approach.** The {lead_name} stays open {qa[lead_col]:.1f}% of the time with the optimized plan, versus {pol[lead_col]:.1f}% "
    f"when the same budget goes to the most-complained-about roads. Expected failures left: {qa['expected_failures_left']:.1f} vs {pol['expected_failures_left']:.1f}.",
    f"- **Honest caveat.** With {n} qubits there are only {2**n:,} possible plans, and 4,000 shots sample most of them. A classical solver is instant here. "
    "This shows the formulation works and that QAOA concentrates probability on good plans. It does not show a quantum speedup.",
]
summary = "\n".join(lines)
(RESULTS / "summary.md").write_text(summary, encoding="utf-8")
display(Markdown(summary))

- **Quantum search vs. random guessing.** On an ideal simulator, 24.5% of QAOA samples are good repair plans (31.4x random guessing). With the noise model of a real IBM chip it falls to 24.0x (light compilation) or 26.1x (heavier compilation, 37 vs 41 two-qubit gates); adding dynamical decoupling gives 25.6x.
- **Best plan found.** With post-selection, the best plan sampled reaches 100% of the optimum on the ideal simulator and 100% under noise.
- **Encoding the budget.** The lean encoding uses 8 qubits and 37 two-qubit gates (31.5x random guessing) versus 12 qubits and 468 gates (2.8x) for the slack encoding.
- **Against the classical baseline.** The quantum search's best plan is the same plan as the exact classical optimum (objective 9.45 vs 9.45).
- **Against today's approach.** The route to the hospital stays open 32.7% of the time with the optimized plan, versus 0.3% when the same budget goes to the most-complained-about roads. Expected failures left: 7.5 vs 8.5.
- **Honest caveat.** With 8 qubits there are only 256 possible plans, and 4,000 shots sample most of them. A classical solver is instant here. This shows the formulation works and that QAOA concentrates probability on good plans. It does not show a quantum speedup.

**Reading the dynamical decoupling result.** DD protects idle qubits from slow, coherent dephasing (for example crosstalk with neighboring qubits). The simulator's noise model mostly contains gate errors, readout errors and T1/T2 relaxation, which DD cannot undo,
so little or no improvement is expected in simulation. On real hardware DD often helps, which is one reason the hardware run in section 9 matters.

### Honest limits
- **Tiny problem.** 8 qubits and 256 possible plans. A laptop solves it exactly in milliseconds. We do not claim a quantum speedup, and QAOA has no guaranteed advantage on problems like this.
- **Noise.** Real chips today are noisy, and deeper circuits lose accuracy quickly. Our circuit is shallow on purpose (`REPS = 2`).
- **Data and model.** The inputs are illustrative, and the risk score is a transparent weighted heuristic with guessed weights, not a trained model. Failures are assumed independent.
- **Not for operations.** This is a prototype, not flood prediction, engineering design or an actual repair plan.

### What we learned
- Writing the budget as a **price per crew-day** (lean) instead of extra slack qubits shrank the circuit and made QAOA far more effective. The exact numbers for this run are in the section 7 table and the summary above.
- A circuit that looks small can grow a lot when transpiled for real hardware, so compilation matters as much as the algorithm.
- Comparing against a classical baseline is what makes the result believable.

### What would have to be true to scale
Measured data (surveyed geometry, elevation, groundwater, tide records, geocoded 311 history, real repair costs), a validated risk model, a county-sized problem with thousands of roads,
and quantum hardware with many more qubits and much lower error rates, or error correction. Next steps: run on more hardware backends, try error mitigation on real runs, and test larger `K`.

### Glossary
**QUBO**: a yes/no choice per road with pairwise bonuses. **Qubit**: a quantum bit. **QAOA**: a hybrid quantum-classical algorithm for picking the best combination.
**Transpilation**: rewriting a circuit for a specific device. **Shot**: one run-and-measure of the circuit. **Noise model**: a simulation of a device's errors. **Post-selection**: discarding invalid results.